# Probability Distribution Fitting, Flood Frequency Estimation, Goodness-of-Fit Analysis, and Visualization
## Overview
This notebook fits four probability distributions to the annual maximum flood-flow series: Normal, Log-Normal, Gumbel, and Log-Pearson Type III. The estimated distribution parameters are used to calculate design flood discharges for selected return periods. The performance of each distribution is evaluated using the Kolmogorov–Smirnov test, Chi-square test, and root mean square error. The distributions are then ranked according to their goodness-of-fit results. Graphical comparisons are also prepared to examine the observed data, fitted probability distributions, flood-frequency curves, design flood estimates, and overall goodness-of-fit ranking.


In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
from IPython.display import display

In [ ]:
# Change this station number for a different study area
SITE_NO = "06214500"

# Define output file paths
OUTPUT_DIR = Path(f"outputs/usgs_{SITE_NO}")
FIGURE_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Clean Flood Dataset

In [ ]:
# Load clean flood dataset
df = pd.read_csv(f"data/processed/usgs_{SITE_NO}_clean_flood_data.csv")

df.head()

In [ ]:
flow = df["Peak_Flow"]

## 2. Fit Probability Distributions

In [ ]:
# Normal Distribution
normal_mu, normal_sigma = stats.norm.fit(flow)

# Log-Normal Distribution
lognorm_shape, lognorm_loc, lognorm_scale = stats.lognorm.fit(flow, floc=0)
lognorm_mu_log = np.log(lognorm_scale)
lognorm_sigma_log = lognorm_shape

# Gumbel Distribution
gumbel_loc, gumbel_scale = stats.gumbel_r.fit(flow)

# Log-Pearson Type III Distribution
log10_flow = np.log10(flow)
lp3_skew, lp3_loc, lp3_scale = stats.pearson3.fit(log10_flow)

In [ ]:
# Combined parameter table
parameters = pd.DataFrame({
    "Distribution": [
        "Normal", "Normal", 
        "Log-Normal", "Log-Normal", "Log-Normal", "Log-Normal", 
        "Gumbel", "Gumbel", 
        "Log-Pearson Type III", "Log-Pearson Type III", "Log-Pearson Type III", "Log-Pearson Type III"
    ],
    "Parameter": [
        "Mean, μ", "Standard Deviation, σ", 
        "Mean of ln(Q), μln", "Standard Deviation of ln(Q), σln", "Shape", "Scale", 
        "Location", "Scale", 
        "Log Base", "Skewness", "Location", "Scale"
    ],
    "Value": [
        normal_mu, normal_sigma, 
        lognorm_mu_log, lognorm_sigma_log, lognorm_shape, lognorm_scale, 
        gumbel_loc, gumbel_scale, 
        "log10", lp3_skew, lp3_loc, lp3_scale
    ]
})

parameters["Value"] = parameters["Value"].apply(lambda x:round(x, 5) if isinstance(x, (int, float, np.floating)) else x)

parameters.to_csv(OUTPUT_DIR / "combined_distribution_parameters.csv", index=False)

display(parameters)

## 3. Flood Frequency Estimation

In [ ]:
return_periods = np.array([2, 5, 10, 25, 50, 100, 200])

exceedance_prob = 1 / return_periods
non_exceedance_prob = 1 - exceedance_prob

In [ ]:
# Normal
normal_q = stats.norm.ppf(non_exceedance_prob, loc=normal_mu, scale=normal_sigma)

# Log-Normal
lognormal_q = stats.lognorm.ppf(non_exceedance_prob, lognorm_shape, loc=lognorm_loc, scale=lognorm_scale)

# Gumbel
gumbel_q = stats.gumbel_r.ppf(non_exceedance_prob, loc=gumbel_loc, scale=gumbel_scale)

# Log-Pearson Type III
lp3_log_q = stats.pearson3.ppf(non_exceedance_prob, lp3_skew, loc=lp3_loc, scale=lp3_scale )
lp3_q = 10 ** lp3_log_q

In [ ]:
# Design flood table
design_flood_table = pd.DataFrame({
    "Return Period (Years)": return_periods, 
    "Exceedance Probability": exceedance_prob, 
    "Non-Exceedance Probability": non_exceedance_prob,
    "Normal": normal_q,
    "Log-Normal": lognormal_q,
    "Gumbel": gumbel_q,
    "Log-Pearson Type III": lp3_q
})

design_flood_table = design_flood_table.round(3)

design_flood_table.to_csv(
    OUTPUT_DIR / "design_flood_estimates.csv",
    index=False
)

display(design_flood_table)

## 4. Goodness-of-Fit Analysis

### 4.1 KS Test

In [ ]:
# cdf functions
def normal_cdf(x):
    return stats.norm.cdf(x, loc=normal_mu, scale=normal_sigma)

def lognormal_cdf(x):
    return stats.lognorm.cdf(x, lognorm_shape, loc=lognorm_loc, scale=lognorm_scale)

def gumbel_cdf(x):
    return stats.gumbel_r.cdf(x, loc=gumbel_loc, scale=gumbel_scale)

def lp3_cdf(x):
    x = np.asarray(x)
    return stats.pearson3.cdf(np.log10(x), lp3_skew, loc=lp3_loc, scale=lp3_scale)

In [ ]:
# ks
ks_results = pd.DataFrame({
    "Distribution":[
        "Normal", 
        "Log-Normal", 
        "Gumbel", 
        "Log-Pearson Type III"
    ],
    "KS Statistic": [
        stats.kstest(flow, normal_cdf)[0], 
        stats.kstest(flow, lognormal_cdf)[0], 
        stats.kstest(flow, gumbel_cdf)[0], 
        stats.kstest(flow, lp3_cdf)[0]
    ],
    "KS p-value": [
        stats.kstest(flow, normal_cdf)[1], 
        stats.kstest(flow, lognormal_cdf)[1], 
        stats.kstest(flow, gumbel_cdf)[1], 
        stats.kstest(flow, lp3_cdf)[1]
    ]
})

### 4.2 RMSE analysis

In [ ]:
# observed
observed_sorted = np.sort(flow)
n = len(observed_sorted)

ranks = np.arange(1, n + 1)
plotting_position = ranks / (n + 1)

# estimated
normal_theoretical = stats.norm.ppf(plotting_position, loc=normal_mu, scale=normal_sigma)

lognormal_theoretical = stats.lognorm.ppf(plotting_position, lognorm_shape, loc=lognorm_loc, scale=lognorm_scale)

gumbel_theoretical = stats.gumbel_r.ppf(plotting_position, loc=gumbel_loc, scale=gumbel_scale)

lp3_log_theoretical = stats.pearson3.ppf(plotting_position, lp3_skew, loc=lp3_loc, scale=lp3_scale)
lp3_theoretical = 10 ** lp3_log_theoretical

# rmse
def rmse(observed, estimated):
    return np.sqrt(np.mean((observed - estimated) ** 2))

rmse_results = pd.DataFrame({
    "Distribution": [
        "Normal", 
        "Log-Normal", 
        "Gumbel", 
        "Log-Pearson Type III"
    ],
    "RMSE": [
        rmse(observed_sorted, normal_theoretical), 
        rmse(observed_sorted, lognormal_theoretical), 
        rmse(observed_sorted, gumbel_theoretical), 
        rmse(observed_sorted, lp3_theoretical)
    ]
})

### 4.3 Chi-square test

In [ ]:
# ppf functions used to create equal-probability bins
def normal_ppf(p):
    return stats.norm.ppf(p, loc=normal_mu, scale=normal_sigma)

def lognormal_ppf(p):
    return stats.lognorm.ppf(p, lognorm_shape, loc=lognorm_loc, scale=lognorm_scale)

def gumbel_ppf(p):
    return stats.gumbel_r.ppf(p, loc=gumbel_loc, scale=gumbel_scale)

def lp3_ppf(p):
    log_quantile = stats.pearson3.ppf(p, lp3_skew, loc=lp3_loc, scale=lp3_scale)
    return 10 ** log_quantile

# chi-square 
def chi_square_test(data, ppf_function, num_params, bins=6):
    data = np.asarray(data)

    # Equal-probability class boundaries
    probabilities = np.linspace(0, 1, bins + 1)
    bin_edges = ppf_function(probabilities)

    observed_freq, _ = np.histogram(data, bins=bin_edges)

    # Equal expected frequency in every bin
    expected_freq = np.full(bins, len(data) / bins)

    chi_stat = np.sum((observed_freq - expected_freq) ** 2 / expected_freq)

    dof = bins - 1 - num_params

    p_value = (stats.chi2.sf(chi_stat, dof) if dof > 0 else np.nan)

    return (chi_stat, p_value, dof, expected_freq.min())


chi_data = []

tests = [
    ("Normal", flow, normal_ppf, 2),
    ("Log-Normal", flow, lognormal_ppf, 2),
    ("Gumbel", flow, gumbel_ppf, 2),
    ("Log-Pearson Type III", flow, lp3_ppf, 3)
]

for name, data, ppf_func, num_params in tests:
    chi_stat, chi_p, dof, min_expected = chi_square_test(data, ppf_func, num_params, bins=6)

    chi_data.append([name, chi_stat, chi_p, dof, min_expected])

chi_results = pd.DataFrame(
    chi_data,
    columns=[
        "Distribution",
        "Chi-Square Statistic",
        "Chi-Square p-value",
        "Degrees of Freedom",
        "Minimum Expected Frequency"
    ]
)

### 4.4 Combine GOF table

In [ ]:
gof_results = (ks_results.merge(chi_results, on="Distribution").merge(rmse_results, on="Distribution"))

gof_results = gof_results.round(5)

gof_results.to_csv(
    OUTPUT_DIR / "goodness_of_fit_results.csv",
    index=False
)

display(gof_results)

### 4.5 Ranking table

In [ ]:
ranking_table = gof_results.copy()

ranking_table["KS Rank"] = ranking_table["KS Statistic"].rank(ascending=True)
ranking_table["Chi-Square Rank"] = ranking_table["Chi-Square Statistic"].rank(ascending=True)
ranking_table["RMSE Rank"] = ranking_table["RMSE"].rank(ascending=True)

ranking_table["Average Rank"] = ranking_table[["KS Rank", "Chi-Square Rank", "RMSE Rank"]].mean(axis=1)

ranking_table = ranking_table.sort_values("Average Rank")

ranking_table.to_csv(
    OUTPUT_DIR / "distribution_ranking.csv",
    index=False
)

display(ranking_table)

## 5. Visualization

### 5.1 Create PDF Curves

In [ ]:
x = np.linspace(flow.min(), flow.max(), 1000)

# Normal
normal_pdf = stats.norm.pdf(x, loc=normal_mu, scale=normal_sigma)

# Log-Normal
lognormal_pdf = stats.lognorm.pdf(x, lognorm_shape, loc=lognorm_loc, scale=lognorm_scale)

# Gumbel
gumbel_pdf = stats.gumbel_r.pdf(x, loc=gumbel_loc, scale=gumbel_scale)

# Log-Pearson Type III
lp3_pdf_log = stats.pearson3.pdf(np.log10(x), lp3_skew, loc=lp3_loc, scale=lp3_scale)
lp3_pdf = lp3_pdf_log / (x * np.log(10))

### 5.2 Histogram of Annual Maximum Flood Discharge

In [ ]:
plt.figure(figsize=(9, 6))

plt.hist(flow, bins=10, edgecolor="black", alpha=0.75)

plt.xlabel("Annual Maximum Flood Discharge (m³/s)")
plt.ylabel("Frequency")
plt.title("Histogram of Annual Maximum Flood Discharge")
plt.grid(True, alpha=0.3)

plt.savefig(FIGURE_DIR / "01_observed_flood_histogram.png", dpi=300, bbox_inches="tight")
plt.show()

### 5.3 Comparison of Fitted Probability Density Curves

In [ ]:
plt.figure(figsize=(10, 6))

plt.hist(flow, bins=10, density=True, edgecolor="black", alpha=0.65, label="Observed Data")

plt.plot(x, normal_pdf, linewidth=2, label="Normal")
plt.plot(x, lognormal_pdf, linewidth=2, label="Log-Normal")
plt.plot(x, gumbel_pdf, linewidth=2, label="Gumbel")
plt.plot(x, lp3_pdf, linewidth=2, label="Log-Pearson Type III")

plt.xlabel("Annual Maximum Flood Discharge (m³/s)")
plt.ylabel("Probability Density")
plt.title("Comparison of Fitted Probability Distributions")
plt.legend()
plt.grid(True, alpha=0.3)

plt.savefig(FIGURE_DIR / "02_distribution_comparison_pdf.png", dpi=300, bbox_inches="tight")
plt.show()

### 5.4 Observed Flood Frequency Curve

In [ ]:
observed_desc = np.sort(flow)[::-1]

n = len(observed_desc)
rank = np.arange(1, n + 1)

exceedance_probability = rank / (n + 1)
observed_return_period = 1 / exceedance_probability

plt.figure(figsize=(9, 6))

plt.scatter(observed_return_period, observed_desc, label="Observed Data")

plt.xscale("log")
plt.xlabel("Return Period (Years)")
plt.ylabel("Annual Maximum Flood Discharge (m³/s)")
plt.title("Observed Flood Frequency Curve")
plt.legend()
plt.grid(True, which="both", alpha=0.3)

plt.savefig(FIGURE_DIR / "03_observed_return_period_plot.png", dpi=300, bbox_inches="tight")
plt.show()

### 5.5 Comparison of Design Flood Estimates

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(design_flood_table["Return Period (Years)"], design_flood_table["Normal"], marker="o", linewidth=2, label="Normal")
plt.plot(design_flood_table["Return Period (Years)"], design_flood_table["Log-Normal"], marker="o", linewidth=2, label="Log-Normal")
plt.plot(design_flood_table["Return Period (Years)"], design_flood_table["Gumbel"], marker="o", linewidth=2, label="Gumbel")
plt.plot(design_flood_table["Return Period (Years)"], design_flood_table["Log-Pearson Type III"], marker="o", linewidth=2, label="Log-Pearson Type III")

plt.xscale("log")
plt.xlabel("Return Period (Years)")
plt.ylabel("Design Flood Discharge (m³/s)")
plt.title("Design Flood Estimate Using Different Distribtutions")
plt.legend()
plt.grid(True, which="both", alpha=0.3)

plt.savefig(FIGURE_DIR / "04_design_flood_comparison_plot.png", dpi=300, bbox_inches="tight")
plt.show()


### 5.6 Observed and Fitted Flood Frequency Curves

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(observed_return_period, observed_desc, label="Observed Data")

plt.plot(return_periods, normal_q, marker="o", linewidth=2, label="Normal")
plt.plot(return_periods, lognormal_q, marker="o", linewidth=2, label="Log-Normal")
plt.plot(return_periods, gumbel_q, marker="o", linewidth=2, label="Gumbel")
plt.plot(return_periods, lp3_q, marker="o", linewidth=2, label="Log-Pearson Type III")

plt.xscale("log")
plt.xlabel("Return Period (Years)")
plt.ylabel("Flood Discharge (m³/s)")
plt.title("Observed and Fitted Flood Frequency Curves")
plt.legend()
plt.grid(True, which="both", alpha=0.3)

plt.savefig(FIGURE_DIR / "05_observed_and_fitted_return_period_plot.png", dpi=300, bbox_inches="tight")
plt.show()

#### For Smooth Visualization Only

In [ ]:
# Smooth return periods for visualization only
return_period_smooth = np.logspace(
    np.log10(1.01),
    np.log10(200),
    300
)

non_exceedance_smooth = 1 - (1 / return_period_smooth)

# Normal
normal_smooth = stats.norm.ppf(
    non_exceedance_smooth,
    loc=normal_mu,
    scale=normal_sigma
)

# Log-Normal
lognormal_smooth = stats.lognorm.ppf(
    non_exceedance_smooth,
    lognorm_shape,
    loc=lognorm_loc,
    scale=lognorm_scale
)

# Gumbel
gumbel_smooth = stats.gumbel_r.ppf(
    non_exceedance_smooth,
    loc=gumbel_loc,
    scale=gumbel_scale
)

# Log-Pearson Type III
lp3_log_smooth = stats.pearson3.ppf(
    non_exceedance_smooth,
    lp3_skew,
    loc=lp3_loc,
    scale=lp3_scale
)

lp3_smooth = 10 ** lp3_log_smooth

In [ ]:
plt.figure(figsize=(10, 6))

# Observed flood values
plt.scatter(
    observed_return_period,
    observed_desc,
    label="Observed Data",
    zorder=5
)

# Smooth fitted curves
plt.plot(
    return_period_smooth,
    normal_smooth,
    linewidth=2,
    label="Normal"
)

plt.plot(
    return_period_smooth,
    lognormal_smooth,
    linewidth=2,
    label="Log-Normal"
)

plt.plot(
    return_period_smooth,
    gumbel_smooth,
    linewidth=2,
    label="Gumbel"
)

plt.plot(
    return_period_smooth,
    lp3_smooth,
    linewidth=2,
    label="Log-Pearson Type III"
)

plt.xscale("log")
plt.xlim(1, 200)

plt.xlabel("Return Period (Years)")
plt.ylabel("Flood Discharge (m³/s)")
plt.title("Observed and Fitted Flood Frequency Curves")
plt.legend()
plt.grid(True, which="both", alpha=0.3)

plt.savefig(
    FIGURE_DIR / "05_smooth_observed_and_fitted_return_period_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### 5.7 Goodness-of-Fit Ranking

In [ ]:
ranking_sorted = ranking_table.sort_values("Average Rank")

plt.figure(figsize=(9, 6))

plt.barh(
    ranking_sorted["Distribution"],
    ranking_sorted["Average Rank"],
    edgecolor="black"
)

plt.xlabel("Average Rank")
plt.ylabel("Distribution")
plt.title("Goodness-of-Fit Ranking")
plt.gca().invert_yaxis()
plt.grid(True, axis="x", alpha=0.3)

plt.savefig(FIGURE_DIR / "06_goodness_of_fit_ranking.png", dpi=300, bbox_inches="tight")
plt.show()